# 🚀 EADF-Net: Google Colab Training & Benchmarking Pipeline
### Beating SGDC-DeepLab (The Visual Computer 2026) across all metrics on LoveDA & FGGS datasets

**Target Metrics to Beat (LoveDA):**
- **mIoU**: > **61.20%**
- **OA**: > **76.97%**
- **mF1**: > **78.59%**
- Lightweight footprint: ~12M parameters


## Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## Step 2: Mount Google Drive (Optional - for persistent checkpoint saving)

In [ ]:
# Optional: Uncomment to save checkpoints directly to your Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

## Step 3: Setup EADF-Net Repository & Dependencies

In [ ]:
import os
# Automatically unpack EADF-Net.zip if uploaded to /content
if os.path.exists('/content/EADF-Net.zip'):
    !unzip -q -o /content/EADF-Net.zip -d /content
    print('Successfully unpacked EADF-Net.zip')

%cd /content/EADF-Net
!pip install -q timm torchvision pyyaml

## Step 4: Multi-Threaded High-Speed Download (aria2c - 16 Parallel Connections)
Standard wget is throttled by Zenodo to ~380KB/s (taking >3 hours).
Using aria2c with 16 parallel threads accelerates the download by 15-30x (down to ~2-3 minutes)!

In [ ]:
# Install aria2 multi-connection download accelerator
!apt-get install -y -qq aria2

!mkdir -p /content/data/LoveDA
%cd /content/data

print('Downloading Train set with 16 parallel streams...')
!aria2c -x 16 -s 16 -k 1M --file-allocation=none "https://zenodo.org/records/5706578/files/Train.zip?download=1" -o loveda_train.zip

print('Downloading Val set with 16 parallel streams...')
!aria2c -x 16 -s 16 -k 1M --file-allocation=none "https://zenodo.org/records/5706578/files/Val.zip?download=1" -o loveda_val.zip

print('Extracting archives...')
!unzip -q -o loveda_train.zip -d /content/data/LoveDA
!unzip -q -o loveda_val.zip -d /content/data/LoveDA
!rm -f loveda_train.zip loveda_val.zip

print('Dataset ready at /content/data/LoveDA!')
%cd /content/EADF-Net

## Step 5: Test Forward Pass & Verify Model Architecture

In [ ]:
import sys
import os
# Ensure Python can find EADF-Net modules from any working directory
for p in ['/content/EADF-Net', '/content', os.getcwd()]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from models import EADFNet

model = EADFNet(num_classes=7, backbone='efficientnet_b1', pretrained_backbone=True)
x = torch.randn(2, 3, 512, 512)
if torch.cuda.is_available():
    model = model.cuda()
    x = x.cuda()

outputs = model(x)
print('Output shape:', outputs['main'].shape)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params / 1e6:.2f}M')

## Step 6: Train EADF-Net
Trains for 100 epochs matching SGDC-DeepLab's protocol (50 frozen backbone + 50 fine-tuning) with mixed-precision (AMP) enabled.

In [ ]:
!python train.py \
    --dataset loveda \
    --data_root /content/data/LoveDA \
    --backbone efficientnet_b1 \
    --epochs 100 \
    --freeze_epochs 50 \
    --batch_size 4 \
    --lr 0.0005 \
    --crop_size 512 \
    --num_workers 4 \
    --use_amp \
    --save_dir /content/checkpoints/loveda

## Step 7: Run Evaluation & Head-to-Head Comparison
Computes mIoU, OA, mF1, and per-class IoU against SGDC-DeepLab and all baselines.

In [ ]:
!python evaluate.py \
    --checkpoint /content/checkpoints/loveda/best_model.pth \
    --data_root /content/data/LoveDA \
    --dataset loveda \
    --batch_size 4 \
    --crop_size 512